# CertVIC remaining runs -- Qwen2.5-VL-7B T4x2 parallel

Runs exactly one remaining CertVIC bundle per Kaggle session: `spurious`,
`perception_scaled`, `polarity`, or `mechanism`.

Default settings: Kaggle **GPU T4 x2**, Internet ON, one provider, one task
bundle, deterministic two-shard execution. With two GPUs, shard0 runs with
`CUDA_VISIBLE_DEVICES=0` and shard1 with `CUDA_VISIBLE_DEVICES=1`. With one GPU,
both shards run sequentially on GPU0. Shard outputs, logs, summary JSON, runtime
manifest, and a download zip are written to `/kaggle/working`.

This notebook prepares predictions only when you run it on Kaggle; the local
builder does not run models and does not create model results.

In [ ]:
# Install provider-specific runtime dependencies before importing transformers.
import os, subprocess, sys
DEPS_SENTINEL = "/kaggle/working/.certvic_deps_qwen2_5_vl_7b"
if not os.path.exists(DEPS_SENTINEL):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'transformers==4.49.0', 'accelerate>=0.34', 'sentencepiece', 'huggingface_hub>=0.24', 'qwen-vl-utils', 'bitsandbytes>=0.45.0'], check=True)
    open(DEPS_SENTINEL, "w").write("ok\n")
    print("installed pinned stack for qwen2_5_vl_7b")
else:
    print("dependency sentinel exists:", DEPS_SENTINEL)

In [ ]:
# Editable configuration and input auto-detection.
import glob, json, os, sys, zipfile
from pathlib import Path

CERTVIC_DIR = None       # parent directory containing certvic/
BUNDLE_INPUT = None      # bundle directory or .zip; auto-detected when None
OUTPUT_DIR = "/kaggle/working"
MODEL_CACHE_DIR = "/kaggle/working/hf_models/qwen2_5_vl_7b"
MODEL_REVISION = None    # REQUIRED: exact 40-character Hugging Face commit SHA
PROVIDER = "qwen2_5_vl_7b"
RUN_TAG = None           # one of: spurious, perception_scaled, polarity, mechanism
ALLOW_INTERNVL_TWO_WORKER = False  # advanced only; default is safer shared T4x2 sequential mode

RUN_TAGS = ['spurious', 'perception_scaled', 'polarity', 'mechanism']
BUNDLE_BY_TAG = {
    "spurious": "certvic_spurious_flip_control.zip",
    "perception_scaled": "certvic_perception_control_scaled.zip",
    "polarity": "certvic_polarity_ablations.zip",
    "mechanism": "certvic_mechanism_probes.zip",
}

def _find_certvic_parent():
    for p in glob.glob("/kaggle/input/**/certvic/eval/run_eval.py", recursive=True):
        return str(Path(p).parents[2])
    return None

def _materialize_zip(path):
    path = str(path)
    if not path.endswith(".zip"):
        return path
    dest = Path(OUTPUT_DIR) / ("bundle_" + Path(path).stem)
    marker = dest / ".unzipped"
    if not marker.exists():
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(path) as zf:
            zf.extractall(dest)
        marker.write_text("ok\n")
    return str(dest)

def _bundle_tag_from_manifest(directory):
    manifest = Path(directory) / "bundle_manifest.json"
    if manifest.exists():
        return json.loads(manifest.read_text()).get("run_tag")
    name = Path(directory).name
    for tag, zip_name in BUNDLE_BY_TAG.items():
        if tag in name or zip_name.replace(".zip", "") in name:
            return tag
    return None

def _find_bundle():
    candidates = []
    for p in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if Path(p).name in BUNDLE_BY_TAG.values():
            candidates.append(p)
    for p in glob.glob("/kaggle/input/**/bundle_manifest.json", recursive=True):
        candidates.append(str(Path(p).parent))
    for name in ("pilot_eval_tasks_reviewed.jsonl", "tasks.jsonl"):
        for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
            candidates.append(str(Path(p).parent))
    tagged = []
    for c in sorted(set(candidates)):
        directory = _materialize_zip(c)
        tag = _bundle_tag_from_manifest(directory)
        if tag:
            tagged.append((tag, directory))
    if RUN_TAG:
        matches = [directory for tag, directory in tagged if tag == RUN_TAG]
        if matches:
            return sorted(matches, key=len)[0], RUN_TAG
    if len(tagged) == 1:
        return tagged[0][1], tagged[0][0]
    raise RuntimeError("Attach exactly one remaining-run task bundle or set BUNDLE_INPUT and RUN_TAG.")

CERTVIC_DIR = CERTVIC_DIR or _find_certvic_parent()
if BUNDLE_INPUT:
    BUNDLE_INPUT = _materialize_zip(BUNDLE_INPUT)
    RUN_TAG = RUN_TAG or _bundle_tag_from_manifest(BUNDLE_INPUT)
else:
    BUNDLE_INPUT, RUN_TAG = _find_bundle()
if RUN_TAG not in RUN_TAGS:
    raise RuntimeError(f"RUN_TAG must be one of {RUN_TAGS}, got {RUN_TAG!r}")
if not CERTVIC_DIR:
    raise RuntimeError("Attach the CertVIC code bundle, e.g. dist/certvic_kaggle_main200_bundle.zip.")

sys.path.insert(0, CERTVIC_DIR)
import certvic
print("CERTVIC_DIR      :", CERTVIC_DIR)
print("BUNDLE_INPUT     :", BUNDLE_INPUT)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("MODEL_CACHE_DIR  :", MODEL_CACHE_DIR)
print("PROVIDER / RUN_TAG:", PROVIDER, RUN_TAG)
print("certvic import   :", certvic.__file__)

In [ ]:
# GPU inventory. Kaggle setting should be Accelerator = GPU T4 x2.
import json, torch
GPU_COUNT = torch.cuda.device_count()
print("torch.cuda.device_count() =", GPU_COUNT)
GPU_INFO = []
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    mem_gb = round(props.total_memory / (1024 ** 3), 2)
    GPU_INFO.append({"index": i, "name": props.name, "memory_gb": mem_gb})
    print(f"GPU {i}: {props.name} | {mem_gb} GiB")
if GPU_COUNT < 1:
    raise RuntimeError("No CUDA GPU detected. In Kaggle choose Accelerator = GPU T4 x2.")
if GPU_COUNT == 1:
    print("WARNING: only one GPU detected; both deterministic shards will run sequentially on GPU0.")
if GPU_COUNT >= 2:
    print("T4x2 path available: worker 0 uses CUDA_VISIBLE_DEVICES=0 and worker 1 uses CUDA_VISIBLE_DEVICES=1.")

In [ ]:
# Download one exact model revision. Internet must be ON for first run.
import os, re
from huggingface_hub import snapshot_download

MODEL_REPO_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
if not isinstance(MODEL_REVISION, str) or not re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION):
    raise RuntimeError(
        "Set MODEL_REVISION in the configuration cell to an exact 40-character "
        "Hugging Face commit SHA before any provider run."
    )

def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
revision_marker = os.path.join(MODEL_CACHE_DIR, ".certvic_model_revision")
if os.path.exists(os.path.join(MODEL_CACHE_DIR, "config.json")):
    if not os.path.exists(revision_marker):
        raise RuntimeError("Cached model has no CertVIC revision marker; use a fresh cache directory.")
    cached_revision = open(revision_marker).read().strip()
    if cached_revision != MODEL_REVISION:
        raise RuntimeError(
            f"Cached model revision {cached_revision!r} != locked {MODEL_REVISION!r}; "
            "use a fresh cache directory."
        )
    print("reusing cached model:", MODEL_CACHE_DIR)
else:
    print("downloading", MODEL_REPO_ID, "revision", MODEL_REVISION, "->", MODEL_CACHE_DIR)
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        revision=MODEL_REVISION,
        local_dir=MODEL_CACHE_DIR,
        token=_hf_token(),
    )
    open(revision_marker, "w").write(MODEL_REVISION + "\n")
assert os.path.exists(os.path.join(MODEL_CACHE_DIR, "config.json")), "model download missing config.json"
assert open(revision_marker).read().strip() == MODEL_REVISION, "model revision marker mismatch"

In [ ]:
# Prepare portable task shards. Control bundles use TaskItem/run_eval; diagnostics use the flat loop.
import json, os
from pathlib import Path

MODE_BY_TAG = {
    "spurious": "control",
    "perception_scaled": "control",
    "polarity": "diagnostic",
    "mechanism": "diagnostic",
}
EXPECTED_ROWS_BY_TAG = {"spurious": 94, "perception_scaled": 369, "polarity": 728, "mechanism": 364}
EXPECTED_PREDS_BY_TAG = {"spurious": 188, "perception_scaled": 738, "polarity": 728, "mechanism": 364}
TASK_FILE_BY_MODE = {"control": "pilot_eval_tasks_reviewed.jsonl", "diagnostic": "tasks.jsonl"}
RUN_MODE = MODE_BY_TAG[RUN_TAG]
TASK_FILE = TASK_FILE_BY_MODE[RUN_MODE]
task_path = Path(BUNDLE_INPUT) / TASK_FILE
if not task_path.exists():
    raise RuntimeError(f"Expected {TASK_FILE} in {BUNDLE_INPUT}")
rows = [json.loads(line) for line in task_path.read_text().splitlines() if line.strip()]
if len(rows) != EXPECTED_ROWS_BY_TAG[RUN_TAG]:
    raise RuntimeError(f"{RUN_TAG}: expected {EXPECTED_ROWS_BY_TAG[RUN_TAG]} rows, found {len(rows)}")

def _remap_control_path(raw, variant):
    base = os.path.basename(str(raw).replace("__CTRL__/", ""))
    if variant == "original":
        return str(Path(BUNDLE_INPUT) / "orig" / base)
    return str(Path(BUNDLE_INPUT) / base)

def _remap_probe_path(raw, role):
    raw = str(raw or "")
    base = os.path.basename(raw.replace("__PROBE__/", ""))
    if role == "original" or "/orig/" in raw or raw.startswith("__PROBE__/orig/"):
        return str(Path(BUNDLE_INPUT) / "orig" / base)
    return str(Path(BUNDLE_INPUT) / base)

prepared = []
missing = []
if RUN_MODE == "control":
    for row in rows:
        r = dict(row)
        r["original_image_path"] = _remap_control_path(r["original_image_path"], "original")
        r["edited_image_path"] = _remap_control_path(r["edited_image_path"], "edited")
        for key in ("original_image_path", "edited_image_path"):
            if not Path(r[key]).exists():
                missing.append(r[key])
        prepared.append(r)
else:
    for idx, row in enumerate(rows):
        family = row.get("probe_family") or row.get("ablation_family")
        if family == "original_vs_edited" or row.get("evidence_status") == "SPEC_BLOCKED":
            raise RuntimeError("SPEC_BLOCKED mechanism family original_vs_edited is excluded and refused.")
        r = dict(row)
        r["_row_index"] = idx
        if r.get("original_image_path"):
            r["original_image_path"] = _remap_probe_path(r["original_image_path"], "original")
        if r.get("edited_image_path"):
            r["edited_image_path"] = _remap_probe_path(r["edited_image_path"], "edited")
        if r.get("image_path"):
            role = str(r.get("image_variant") or r.get("image_role") or "").lower()
            r["image_path"] = _remap_probe_path(r["image_path"], role)
        img = r.get("image_path")
        if not img:
            role = str(r.get("image_role") or "edited").lower()
            img = r.get("original_image_path") if role == "original" else r.get("edited_image_path")
        if not img or not Path(img).exists():
            missing.append(str(img))
        prepared.append(r)
if missing:
    raise RuntimeError(f"Missing {len(missing)} referenced image files, first={missing[:3]}")

ROW_ORDER = {row["item_id"]: i for i, row in enumerate(prepared)}
shards = {0: [], 1: []}
for i, row in enumerate(prepared):
    shards[i % 2].append(row)

SHARD_TASKS = {}
SHARD_EXPECT = {}
for shard in (0, 1):
    dst = Path(OUTPUT_DIR) / f"tasks_{PROVIDER}_{RUN_TAG}_shard{shard}.jsonl"
    dst.write_text("\n".join(json.dumps(row, sort_keys=True) for row in shards[shard]) + "\n")
    SHARD_TASKS[shard] = str(dst)
    SHARD_EXPECT[shard] = (2 * len(shards[shard])) if RUN_MODE == "control" else len(shards[shard])
    print(f"shard{shard}: {len(shards[shard])} task rows -> expected {SHARD_EXPECT[shard]} predictions")
print("RUN_MODE:", RUN_MODE, "| expected merged predictions:", EXPECTED_PREDS_BY_TAG[RUN_TAG])

CFG = str(Path(OUTPUT_DIR) / f"kaggle_{PROVIDER}_{RUN_TAG}.yaml")
Path(CFG).write_text(
    "mode: kaggle_open_vlm\n"
    f"provider_name: {PROVIDER}\n"
    f"provider: {PROVIDER}\n"
    f"model_id: {MODEL_CACHE_DIR}\n"
    f"model_version: {MODEL_REVISION}\n"
    "device: cuda\n"
    "dtype: bfloat16\n"
    "batch_size: 1\n"
    "max_new_tokens: 16\n"
    "temperature: 0.0\n"
    "paid_services_enabled: false\n"
)
print("wrote config:", CFG)

In [ ]:
# Write the subprocess worker. Each worker owns one shard and one CUDA visibility mask.
from pathlib import Path
WORKER_CODE = r"""
import json, os, sys, time, traceback
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
sys.path.insert(0, os.environ["CERTVIC_DIR"])

PROVIDER = os.environ["PROVIDER"]
RUN_TAG = os.environ["RUN_TAG"]
RUN_MODE = os.environ["RUN_MODE"]
MODEL_DIR = os.environ["MODEL_CACHE_DIR"]
MODEL_REVISION = os.environ["MODEL_REVISION"]
TASKS_SHARD = os.environ["TASKS_SHARD"]
OUT_PATH = os.environ["OUT_PATH"]
CFG = os.environ["CFG"]
SHARD = int(os.environ["SHARD"])

import torch
from PIL import Image
from certvic.eval.parse import parse_answer
import certvic.providers.open_vlm as ovlm

def _progress(state):
    state["n"] += 1
    if state["n"] <= 2 or state["n"] % 20 == 0:
        dt = max(time.time() - state["t0"], 1e-6)
        print(f"[{PROVIDER} {RUN_TAG} shard{SHARD}] {state['n']} generations | {state['n'] / dt:.3f} gen/s", flush=True)

def _load_qwen():
    from transformers import AutoProcessor, BitsAndBytesConfig
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration as Model
    except Exception:
        from transformers import AutoModelForImageTextToText as Model
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = Model.from_pretrained(MODEL_DIR, device_map={"": 0}, quantization_config=bnb,
                                  low_cpu_mem_usage=True).eval()
    processor = AutoProcessor.from_pretrained(MODEL_DIR, max_pixels=768 * 768)
    state = {"n": 0, "t0": time.time()}

    @torch.inference_mode()
    def answer(self, image_path, prompt):
        image = Image.open(image_path).convert("RGB")
        msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        _progress(state)
        return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return answer

def _load_llava():
    import transformers
    from transformers import AutoProcessor, BitsAndBytesConfig, LlavaOnevisionForConditionalGeneration
    transformers.logging.set_verbosity_error()
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    if hasattr(processor, "image_processor"):
        try:
            processor.image_processor.image_grid_pinpoints = [[384, 384]]
            print("set processor.image_processor.image_grid_pinpoints = [[384, 384]]", flush=True)
        except Exception as exc:
            print("image_grid_pinpoints not set:", repr(exc), flush=True)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(
        MODEL_DIR, device_map={"": 0}, quantization_config=bnb, low_cpu_mem_usage=True
    ).eval()
    pad_token_id = processor.tokenizer.eos_token_id
    state = {"n": 0, "t0": time.time()}

    @torch.inference_mode()
    def answer(self, image_path, prompt):
        image = Image.open(image_path).convert("RGB")
        image.thumbnail((384, 384))
        conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(conv, add_generation_prompt=True)
        inputs = processor(images=image, text=text, return_tensors="pt").to(model.device, torch.float16)
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=pad_token_id)
        _progress(state)
        return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer

def _load_internvl():
    import math
    import torchvision.transforms as T
    from torchvision.transforms.functional import InterpolationMode
    from transformers import AutoModel, AutoTokenizer

    IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    MAX_TILES = 1

    def build_transform(size):
        return T.Compose([
            T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
            T.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def split_model(num_layers=32):
        n = torch.cuda.device_count()
        if n <= 1:
            return "auto"
        per = math.ceil(num_layers / (n - 0.5))
        alloc = [per] * n
        alloc[0] = math.ceil(per * 0.5)
        device_map, layer = {}, 0
        for gpu, count in enumerate(alloc):
            for _ in range(count):
                if layer < num_layers:
                    device_map[f"language_model.model.layers.{layer}"] = gpu
                    layer += 1
        for key in ["vision_model", "mlp1", "language_model.model.tok_embeddings",
                    "language_model.model.embed_tokens", "language_model.output",
                    "language_model.model.norm", "language_model.model.rotary_emb",
                    "language_model.lm_head", f"language_model.model.layers.{num_layers - 1}"]:
            device_map[key] = 0
        return device_map

    def load_image(path, input_size=448):
        img = Image.open(path).convert("RGB")
        img.thumbnail((448, 448))
        transform = build_transform(input_size)
        return torch.stack([transform(img)])

    ngpu = torch.cuda.device_count()
    kwargs = dict(torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True)
    if ngpu >= 2:
        print("InternVL shared T4x2 mode: bf16 split_model/device_map, no BitsAndBytesConfig.", flush=True)
        model = AutoModel.from_pretrained(MODEL_DIR, device_map=split_model(), **kwargs).eval()
    else:
        print("InternVL single visible GPU fallback: device_map='auto' with CPU offload; no bitsandbytes/triton.", flush=True)
        offload = "/kaggle/working/internvl_offload"
        os.makedirs(offload, exist_ok=True)
        model = AutoModel.from_pretrained(
            MODEL_DIR, device_map="auto", max_memory={0: "14GiB", "cpu": "32GiB"},
            offload_folder=offload, **kwargs
        ).eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, use_fast=False)
    gen = dict(max_new_tokens=16, do_sample=False)
    state = {"n": 0, "t0": time.time()}

    @torch.inference_mode()
    def answer(self, image_path, prompt):
        pv = load_image(image_path).to(torch.bfloat16).cuda()
        raw = str(model.chat(tokenizer, pv, "<image>\n" + prompt, gen)).strip()
        _progress(state)
        return raw
    return answer

def _patch_provider():
    if PROVIDER == "qwen2_5_vl_7b":
        answer = _load_qwen()
    elif PROVIDER == "llava_onevision_7b":
        answer = _load_llava()
    elif PROVIDER == "internvl_8b":
        answer = _load_internvl()
    else:
        raise RuntimeError(f"unsupported provider {PROVIDER}")
    def _mark_loaded(self):
        self.model_version = MODEL_REVISION
    ovlm.OpenVLMProvider.load = _mark_loaded
    ovlm.OpenVLMProvider.answer = answer

def _prediction_id(row):
    family = row.get("probe_family") or row.get("ablation_family") or "na"
    variant = row.get("image_variant") or row.get("image_role") or "na"
    return f"{RUN_TAG}:{row.get('_row_index')}:{family}:{row.get('item_id')}:{variant}"

def _diagnostic_loop():
    rows = [json.loads(line) for line in Path(TASKS_SHARD).read_text().splitlines() if line.strip()]
    done = set()
    out = Path(OUT_PATH)
    if out.exists():
        for line in out.read_text().splitlines():
            if line.strip():
                done.add(json.loads(line).get("prediction_id"))
    with out.open("a", encoding="utf-8") as handle:
        for row in rows:
            pid = _prediction_id(row)
            if pid in done:
                continue
            family = row.get("probe_family") or row.get("ablation_family")
            if family == "original_vs_edited" or row.get("evidence_status") == "SPEC_BLOCKED":
                raise RuntimeError("Refusing SPEC_BLOCKED diagnostic family original_vs_edited.")
            role = str(row.get("image_variant") or row.get("image_role") or "edited").lower()
            image_path = row.get("image_path")
            if not image_path:
                image_path = row.get("original_image_path") if role == "original" else row.get("edited_image_path")
            prompt = row.get("prompt") or row.get("question") or row.get("question_original")
            raw = ovlm.OpenVLMProvider.answer(None, image_path, prompt)
            parsed = parse_answer(raw, row.get("answer_format", "yes_no"), strict=True)
            record = {
                "prediction_id": pid,
                "_row_index": row.get("_row_index"),
                "run_id": f"remaining_{PROVIDER}_{RUN_TAG}_shard{SHARD}",
                "item_id": row["item_id"],
                "image_variant": role,
                "provider_name": PROVIDER,
                "provider_type": "open_local",
                "model_name": MODEL_DIR,
                "model_version": MODEL_REVISION,
                "prompt": prompt,
                "raw_output": raw,
                "parsed_answer": parsed.parsed_answer,
                "parse_ok": parsed.parse_ok,
                "parse_confidence": parsed.parse_confidence,
                "latency_s": 0.0,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "evidence_status": "DIAGNOSTIC_NON_EVIDENCE",
                "metadata": {
                    "probe_family": row.get("probe_family"),
                    "ablation_family": row.get("ablation_family"),
                    "polarity": row.get("polarity"),
                    "gold_answer": row.get("gold_answer"),
                    "base_gold": row.get("base_gold"),
                    "shard": SHARD,
                },
            }
            handle.write(json.dumps(record, sort_keys=True) + "\n")
            handle.flush()

def main():
    print("worker starting", json.dumps({
        "provider": PROVIDER,
        "run_tag": RUN_TAG,
        "run_mode": RUN_MODE,
        "shard": SHARD,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "torch_cuda_device_count": torch.cuda.device_count(),
    }, sort_keys=True), flush=True)
    _patch_provider()
    if RUN_MODE == "control":
        from certvic.eval.run_eval import run_eval
        run_prefix = "v9" if RUN_TAG == "spurious_v2" else "remaining"
        summary = run_eval(
            config_path=CFG,
            tasks_path=TASKS_SHARD,
            out_path=OUT_PATH,
            provider_name=PROVIDER,
            run_id=f"{run_prefix}_{PROVIDER}_{RUN_TAG}_shard{SHARD}",
            num_shards=1,
            strict_leakage=True,
            evidence_run=(RUN_TAG != "spurious_v2"),
            fail_fast=False,
            overwrite=False,
        )
        print(json.dumps(summary, sort_keys=True), flush=True)
    else:
        _diagnostic_loop()

if __name__ == "__main__":
    try:
        main()
    except Exception:
        traceback.print_exc()
        raise
"""
Path("/kaggle/working/certvic_vlm_worker.py").write_text(WORKER_CODE)
print("wrote /kaggle/working/certvic_vlm_worker.py")

In [ ]:
# Launch workers, resume complete shards, merge deterministically, and zip outputs.
import collections, hashlib, json, os, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

OUTDIR = Path(OUTPUT_DIR)
WORKER = "/kaggle/working/certvic_vlm_worker.py"
OUTDIR.mkdir(parents=True, exist_ok=True)

def count_jsonl(path):
    p = Path(path)
    if not p.exists():
        return 0
    return sum(1 for line in p.read_text().splitlines() if line.strip())

def shard_out(shard):
    return str(OUTDIR / f"pred_{PROVIDER}_{RUN_TAG}_shard{shard}.jsonl")

def shard_log(shard):
    return str(OUTDIR / f"log_{PROVIDER}_{RUN_TAG}_shard{shard}.txt")

def shard_complete(shard):
    n = count_jsonl(shard_out(shard))
    return n == SHARD_EXPECT[shard]

def launch(shard, visible_devices):
    env = dict(os.environ)
    env.update({
        "CUDA_VISIBLE_DEVICES": str(visible_devices),
        "CERTVIC_DIR": CERTVIC_DIR,
        "MODEL_CACHE_DIR": MODEL_CACHE_DIR,
        "MODEL_REVISION": MODEL_REVISION,
        "PROVIDER": PROVIDER,
        "RUN_TAG": RUN_TAG,
        "RUN_MODE": RUN_MODE,
        "TASKS_SHARD": SHARD_TASKS[shard],
        "OUT_PATH": shard_out(shard),
        "CFG": CFG,
        "SHARD": str(shard),
        "PYTHONUNBUFFERED": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    })
    log_handle = open(shard_log(shard), "a", encoding="utf-8")
    log_handle.write(f"\n=== launch {datetime.now(timezone.utc).isoformat()} CUDA_VISIBLE_DEVICES={visible_devices} ===\n")
    log_handle.flush()
    return subprocess.Popen([sys.executable, "-u", WORKER], env=env, stdout=log_handle, stderr=subprocess.STDOUT), log_handle

def wait_for(procs):
    t0 = time.time()
    while any(proc.poll() is None for proc, _handle in procs.values()):
        time.sleep(20)
        parts = []
        for shard, (proc, _handle) in sorted(procs.items()):
            status = "run" if proc.poll() is None else f"exit={proc.returncode}"
            parts.append(f"shard{shard}:{count_jsonl(shard_out(shard))}/{SHARD_EXPECT[shard]} {status}")
        print(f"[{int(time.time() - t0):4d}s {PROVIDER} {RUN_TAG}] " + " | ".join(parts), flush=True)
    for _proc, handle in procs.values():
        handle.close()
    failures = {shard: proc.returncode for shard, (proc, _handle) in procs.items() if proc.returncode != 0}
    if failures:
        for shard in failures:
            log = Path(shard_log(shard)).read_text(errors="replace")[-2000:]
            print(f"--- log tail shard{shard} ---\n{log}", flush=True)
        raise RuntimeError(f"worker failure(s): {failures}")

def run_needed_shards():
    needed = [s for s in (0, 1) if not shard_complete(s)]
    if not needed:
        print("all shard outputs already complete; skipping workers")
        return
    if PROVIDER == "internvl_8b" and GPU_COUNT >= 2 and not ALLOW_INTERNVL_TWO_WORKER:
        print("InternVL auto-fallback: two full model copies are not memory-safe on T4x2 without bitsandbytes/triton.")
        print("Running shard0 then shard1 with CUDA_VISIBLE_DEVICES=0,1 and device_map/split_model across both GPUs.")
        for shard in needed:
            proc, handle = launch(shard, "0,1")
            wait_for({shard: (proc, handle)})
        return
    if GPU_COUNT >= 2:
        print("Launching two parallel GPU workers: shard0 -> CUDA_VISIBLE_DEVICES=0, shard1 -> CUDA_VISIBLE_DEVICES=1")
        procs = {}
        for shard in needed:
            visible = "0" if shard == 0 else "1"
            procs[shard] = launch(shard, visible)
        wait_for(procs)
    else:
        print("WARNING: single-GPU fallback; running both shards sequentially on CUDA_VISIBLE_DEVICES=0.")
        for shard in needed:
            proc, handle = launch(shard, "0")
            wait_for({shard: (proc, handle)})

runtime_manifest = {
    "schema": "certvic.kaggle_remaining_runtime_manifest.v1",
    "provider": PROVIDER,
    "run_tag": RUN_TAG,
    "run_mode": RUN_MODE,
    "model_repo_id": MODEL_REPO_ID,
    "model_revision": MODEL_REVISION,
    "model_revision_marker_verified": True,
    "code_bundle_sha256": globals().get("CODE_BUNDLE_SHA256"),
    "control_bundle_sha256": globals().get("CONTROL_BUNDLE_SHA256"),
    "gpu_count_detected": GPU_COUNT,
    "gpu_info": GPU_INFO,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "paper_evidence": False,
    "produced_model_results_by_notebook_build": False,
    "shard_expected_rows": SHARD_EXPECT,
}
run_needed_shards()
runtime_manifest["finished_utc"] = datetime.now(timezone.utc).isoformat()

for shard in (0, 1):
    actual = count_jsonl(shard_out(shard))
    if actual != SHARD_EXPECT[shard]:
        raise RuntimeError(f"shard{shard} incomplete: expected {SHARD_EXPECT[shard]}, found {actual}")

variant_order = {"original": 0, "edited": 1}
records = []
seen = set()
for shard in (0, 1):
    for line in Path(shard_out(shard)).read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        if RUN_MODE == "control":
            key = (rec.get("item_id"), rec.get("image_variant"))
            sort_key = (ROW_ORDER.get(rec.get("item_id"), 10**12), variant_order.get(rec.get("image_variant"), 9))
        else:
            key = rec.get("prediction_id")
            sort_key = (int(rec.get("_row_index", 10**12)), str(rec.get("prediction_id")))
        if key in seen:
            raise RuntimeError(f"duplicate prediction id: {key}")
        seen.add(key)
        records.append((sort_key, rec))
records.sort(key=lambda item: item[0])
expected_total = EXPECTED_PREDS_BY_TAG[RUN_TAG]
if len(records) != expected_total:
    raise RuntimeError(f"merged row count mismatch: expected {expected_total}, found {len(records)}")
if RUN_MODE == "control":
    expected_keys = {(row["item_id"], variant) for row in prepared for variant in ("original", "edited")}
    observed_keys = {(rec.get("item_id"), rec.get("image_variant")) for _key, rec in records}
    if observed_keys != expected_keys:
        missing = sorted(expected_keys - observed_keys)
        extra = sorted(observed_keys - expected_keys)
        raise RuntimeError(f"prediction key mismatch: missing={missing[:5]} extra={extra[:5]}")
    wrong_provider = [rec.get("provider_name") for _key, rec in records if rec.get("provider_name") != PROVIDER]
    if wrong_provider:
        raise RuntimeError(f"provider mismatch in merged rows: {sorted(set(wrong_provider))}")
    parse_failures = [
        (rec.get("item_id"), rec.get("image_variant"))
        for _key, rec in records
        if rec.get("parse_ok") is not True or rec.get("parsed_answer") not in {"yes", "no"}
    ]
    if parse_failures:
        raise RuntimeError(
            f"certification-critical parse failures block packaging: {parse_failures[:5]} "
            f"(n={len(parse_failures)})"
        )
merged_name = f"pred_{PROVIDER}_{RUN_TAG}_merged.jsonl" if RUN_MODE == "control" else f"pred_{PROVIDER}_{RUN_TAG}.jsonl"
merged_path = OUTDIR / merged_name
merged_path.write_text("\n".join(json.dumps(rec, sort_keys=True) for _key, rec in records) + "\n")

if RUN_TAG == "spurious_v2":
    runtime_manifest.update({
        "schema": "certvic.v11.spurious_v2.kaggle_output_manifest.v3",
        "expected_items": len(prepared),
        "expected_prediction_rows": expected_total,
        "merged_predictions": merged_name,
        "merged_predictions_sha256": hashlib.sha256(merged_path.read_bytes()).hexdigest(),
        "task_file_sha256": hashlib.sha256(Path(task_path).read_bytes()).hexdigest(),
        "canonical_results_changed": False,
    })

parse_values = [rec.get("parse_ok") for _key, rec in records if "parse_ok" in rec]
answer_counts = collections.Counter(str(rec.get("parsed_answer")) for _key, rec in records if "parsed_answer" in rec)
summary = {
    "schema": "certvic.kaggle_remaining_summary.v1",
    "provider": PROVIDER,
    "run_tag": RUN_TAG,
    "run_mode": RUN_MODE,
    "merged_file": merged_name,
    "expected_rows": expected_total,
    "actual_rows": len(records),
    "duplicates": 0,
    "parse_ok_count": sum(1 for value in parse_values if bool(value)),
    "parse_total": len(parse_values),
    "parse_ok_rate": (sum(1 for value in parse_values if bool(value)) / len(parse_values)) if parse_values else None,
    "answer_counts": dict(answer_counts),
    "paper_evidence": False,
}
summary_path = OUTDIR / f"summary_{PROVIDER}_{RUN_TAG}.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
runtime_path = OUTDIR / f"runtime_manifest_{PROVIDER}_{RUN_TAG}.json"
runtime_path.write_text(json.dumps(runtime_manifest, indent=2, sort_keys=True) + "\n")

zip_path = OUTDIR / f"{PROVIDER}_{RUN_TAG}_preds.zip"
include = [merged_path, summary_path, runtime_path]
for shard in (0, 1):
    include += [Path(shard_out(shard)), Path(shard_log(shard))]
    for suffix in (".run_manifest.json", ".provider_metadata.json", ".environment.json"):
        sidecar = Path(shard_out(shard) + suffix)
        if sidecar.exists():
            include.append(sidecar)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in include:
        if path.exists() and "hf_models" not in str(path):
            zf.write(path, path.name)
print("MERGED:", merged_path)
print("SUMMARY:", summary_path)
print("DOWNLOAD:", zip_path)

## Local ingest
After downloading `<provider>_<run_tag>_preds.zip`, use `LOCAL_INGEST_COMMANDS.md`.